In [11]:
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
import pandas as pd

from rdkit import Chem
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from sklearn.preprocessing import StandardScaler

In [6]:
df = pd.read_csv("final_result2.0_zero_filled.csv")

In [3]:
torch.manual_seed(12345)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(12345)

In [4]:
TARGET_COLS = [
    "CrystalSolub_1", "Molecular Weight", "Log P", "BoilingPoint",
    "is_aromatic", "sigma_780nm", "max_sigma", "Tox-score", "SAscore", "ISC(S1-T1)"
]  # matches your code :contentReference[oaicite:0]{index=0}

In [9]:
# Preprocess SMILES strings to generate graphs using RDKit and PyG
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print("Invalid SMILES string")
        return None
    # Generate the graph (atom features and bond information)
    num_atoms = mol.GetNumAtoms()
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append(atom.GetAtomicNum())
    
    # Create a bond list (edges)
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])  # Because bonds are bidirectional
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    x = torch.tensor(atom_features, dtype=torch.float).view(-1, 1)  # Atom features
    
    return Data(x=x, edge_index=edge_index)

In [12]:

# Build a PyG Dataset (so we can do dataset.shuffle() and slicing)
class GraphListDataset(Dataset):
    def __init__(self, data_list):
        super().__init__()
        self.data_list = data_list

    def len(self):
        return len(self.data_list)

    def get(self, idx):
        return self.data_list[idx]

data_list = []
for _, row in df.iterrows():
    g = smiles_to_graph(row["SMILES"])
    if g is None:
        print("SMILES does not exist!")
        continue

    y_raw = row[TARGET_COLS].to_numpy(dtype = np.float32)  # shape (10,)
    g.y_raw = torch.tensor(y_raw, dtype=torch.float32)   # store unnormalized targets
    data_list.append(g)

dataset = GraphListDataset(data_list)
print(f"Total graphs: {len(dataset)}")

Total graphs: 211137


In [ ]:
# 2) inspect one graph's tensor shapes
g0 = dataset[211136]
print("graph[0].x shape:", tuple(g0.x.shape))                 # (num_nodes, num_node_features)
print("graph[0].edge_index shape:", tuple(g0.edge_index.shape))# (2, num_edges*2) if bidirectional
print("graph[0].y_raw shape:", tuple(g0.y_raw.shape))          # (len(TARGET_COLS),)

graph[0].x shape: (14, 1)
graph[0].edge_index shape: (2, 28)
graph[0].y_raw shape: (10,)


In [21]:
# Colab split style: shuffle + slice (95/5)
torch.manual_seed(12345)
dataset = dataset.shuffle()

train_cut = int(0.95 * len(dataset))
train_dataset = dataset[:train_cut]
test_dataset = dataset[train_cut:]

print(f"Number of training graphs: {len(train_dataset)}")
print(f"Number of test graphs: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Number of training graphs: 200580
Number of test graphs: 10557


In [33]:
# Normalization (fit on TRAIN ONLY), then set data.y for all graphs
scaler = StandardScaler()

# Fit scaler on training y_raw only
train_y = torch.stack([d.y_raw for d in train_dataset], dim=0).cpu().numpy()  # (N_train, 10)
scaler.fit(train_y)

# Transform and attach normalized targets to each Data object (as data.y)
# Important: do this for ALL graphs (train + test) using the train-fitted scaler

# for d in dataset:
    # y_norm = scaler.transform(d.y_raw.view(1, -1).cpu().numpy()).astype(np.float32)[0]  # (10,)
    # d.y = torch.tensor(y_norm, dtype=torch.float32)

for d in dataset:
    y_norm = scaler.transform(d.y_raw.view(1, -1).cpu().numpy()).astype(np.float32)[0]
    d.y = torch.tensor(y_norm, dtype=torch.float32).view(1, -1)  # (1, 10) not (10,)

In [35]:
# sanity check
print("train_y shape:", train_y.shape)          # should be (N_train, 10)
print("scaler.mean_.shape:", scaler.mean_.shape)  # (10,)
print("scaler.scale_.shape:", scaler.scale_.shape) # (10,)
print("first 10 means:", scaler.mean_[:10])
print("first 10 stds:", scaler.scale_[:10])

train_y shape: (200580, 10)
scaler.mean_.shape: (10,)
scaler.scale_.shape: (10,)
first 10 means: [6.67695566e+04 1.61423663e+02 9.86301171e-01 2.47991164e+02
 3.73731180e-01 1.21532906e+02 1.51204294e+02 5.95090493e-01
 1.22190016e-01 1.07263596e-01]
first 10 stds: [1.58218325e+05 6.90586085e+01 1.49605160e+00 1.41683492e+02
 4.83793535e-01 1.00123960e+02 1.15409155e+02 9.47133455e-02
 1.20810717e-01 8.75483930e-02]


In [36]:
# sanity check
train_y_norm = scaler.transform(train_y)
print("col means (train, after):", train_y_norm.mean(axis=0))
print("col stds  (train, after):", train_y_norm.std(axis=0))

col means (train, after): [-1.2097577e-07  9.3376450e-08 -9.1579215e-09 -1.3187669e-07
  2.7199900e-07 -2.2402615e-07 -9.0563816e-08 -1.9862867e-08
  1.1828363e-08  4.3118365e-07]
col stds  (train, after): [0.9996213  1.0002534  0.99998045 0.9999417  0.99895066 0.99985194
 1.0001537  0.99997544 0.9999852  0.99982744]


In [27]:
# Model (Colab-like GCN, but regression output=10)
class GCNRegressor(torch.nn.Module):
    def __init__(self, hidden_channels=64, out_dim=10):
        super().__init__()
        torch.manual_seed(12345)
        self.conv1 = GCNConv(1, hidden_channels)                 # node feature dim = 1 (atomic number)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, out_dim)           # 10 properties

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index)

        x = global_mean_pool(x, batch)                           # (num_graphs_in_batch, hidden_channels)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.lin(x)                                          # (num_graphs_in_batch, 10)
        return x

In [ ]:
# Prepare the data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCNRegressor(hidden_channels=64, out_dim=10).to(device)
learning_rate = 0.001
momentum = 0.9
L2reg = 0.0005
#optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
optimizer = optim.SGD(model.parameters(), lr = learning_rate, momentum = momentum, weight_decay = L2reg)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10, threshold=1e-4, threshold_mode='rel')

criterion = nn.MSELoss()

print(model)

GCNRegressor(
  (conv1): GCNConv(1, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 64)
  (lin): Linear(in_features=64, out_features=10, bias=True)
)


In [38]:
# ----------------------------
# Train / Test loops (regression)
# ----------------------------
def train_epoch():
    model.train()
    total_loss = 0.0
    n = 0
    for data in train_loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)

# used for MSE of test_loader
@torch.no_grad()
def eval_mse(loader):
    model.eval()
    total_loss = 0.0
    n = 0
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        total_loss += loss.item()
        n += 1
    return total_loss / max(n, 1)

num_epochs = 300
for epoch in range(1, num_epochs + 1):
    train_mse = train_epoch()
    if epoch == 1 or epoch % 1 == 0:
        test_mse = eval_mse(test_loader)
        print(f"Epoch: {epoch:03d}, Train MSE: {train_mse:.6f}, Test MSE: {test_mse:.6f}")

Epoch: 001, Train MSE: 0.807604, Test MSE: 0.760211
Epoch: 002, Train MSE: 0.802162, Test MSE: 0.755003
Epoch: 003, Train MSE: 0.795327, Test MSE: 0.756879
Epoch: 004, Train MSE: 0.788322, Test MSE: 0.740484
Epoch: 005, Train MSE: 0.783636, Test MSE: 0.729882
Epoch: 006, Train MSE: 0.775181, Test MSE: 0.730555
Epoch: 007, Train MSE: 0.769361, Test MSE: 0.730885
Epoch: 008, Train MSE: 0.765265, Test MSE: 0.726564
Epoch: 009, Train MSE: 0.760753, Test MSE: 0.717278
Epoch: 010, Train MSE: 0.756536, Test MSE: 0.716005
Epoch: 011, Train MSE: 0.755078, Test MSE: 0.718448
Epoch: 012, Train MSE: 0.752238, Test MSE: 0.705807
Epoch: 013, Train MSE: 0.751031, Test MSE: 0.708955
Epoch: 014, Train MSE: 0.749043, Test MSE: 0.697808
Epoch: 015, Train MSE: 0.749187, Test MSE: 0.701642
Epoch: 016, Train MSE: 0.746486, Test MSE: 0.696242
Epoch: 017, Train MSE: 0.745882, Test MSE: 0.701106
Epoch: 018, Train MSE: 0.744669, Test MSE: 0.706809
Epoch: 019, Train MSE: 0.744562, Test MSE: 0.701552
Epoch: 020, 

In [39]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "target_cols": TARGET_COLS,
    },
    "gnn_regressor_10props_norm.pth"
)

In [40]:

# Inverse-transform predictions back to original units
@torch.no_grad()
def predict_original_units(loader):
    model.eval()
    preds = []
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.batch).cpu().numpy()
        out_orig = out * scaler.scale_ + scaler.mean_
        preds.append(out_orig)
    return np.concatenate(preds, axis=0)

In [45]:
# Load the trained model and scaler
import torch

checkpoint = torch.load(
    "gnn_regressor_10props_norm.pth",
    map_location="cpu",
    weights_only=False,   # <- important
)

model = GCNRegressor(hidden_channels=64, out_dim=10)  # Adjust based on your model's config
model.load_state_dict(checkpoint["model_state_dict"])
scaler_mean = checkpoint["scaler_mean"]
scaler_scale = checkpoint["scaler_scale"]
TARGET_COLS = checkpoint["target_cols"]

# Set model to eval mode and move to correct device
import torch
import numpy as np

# Set the model to evaluation mode
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Select a test sample (for example, the first one)
d = test_dataset[0].to(device)

# Perform the prediction (normalized)
with torch.no_grad():
    pred_norm = model(d.x, d.edge_index, d.batch if hasattr(d, "batch") else torch.zeros(d.x.size(0), dtype=torch.long, device=device))
    pred_norm = pred_norm.detach().cpu().numpy().reshape(1, -1)  # (1, 10)

# Inverse-transform the predictions (to get them back in original units)
pred_orig = scaler.inverse_transform(pred_norm)  # (1, 10)
pred_orig = np.clip(pred_orig, 0, None)  # Clip to avoid negative values if needed

# Get the true properties of the molecule (the target values)
true_norm = d.y.cpu().numpy().reshape(1, -1)  # Normalized target

# Denormalize the true properties
true_orig = scaler.inverse_transform(true_norm)  # (1, 10)
true_orig = np.clip(true_orig, 0, None)  # Clip if needed

# Print the true vs predicted values
print("True properties (denormalized):", true_orig[0])
print("Predicted properties (denormalized):", pred_orig[0])

True properties (denormalized): [2.0950950e+05 1.2517100e+02 3.0170000e-01 1.7862000e+02 1.3454876e-08
 7.4505684e+01 9.4051399e+01 5.9044403e-01 1.6533026e-02 1.8793400e-01]
Predicted properties (denormalized): [7.1477383e+04 1.4747127e+02 9.1261375e-01 2.4055342e+02 3.2412130e-01
 1.2214832e+02 1.5196767e+02 5.9524131e-01 1.1715671e-01 1.2494974e-01]


In [49]:
model.eval()
device = next(model.parameters()).device

d = test_dataset[0].to(device)
batch_vec = torch.zeros(d.num_nodes, dtype=torch.long, device=device)

with torch.no_grad():
    pred_norm = model(d.x, d.edge_index, batch_vec).cpu().numpy().reshape(1, -1)

true_norm = d.y.cpu().numpy().reshape(1, -1)

pred_orig = scaler.inverse_transform(pred_norm)
true_orig = scaler.inverse_transform(true_norm)

# keep 4 digits after decimal
pred_orig = np.round(pred_orig, 2)
true_orig = np.round(true_orig, 2)

print("Targets:", TARGET_COLS)
print("True (orig, 4dp):", true_orig[0])
print("Pred (orig, 4dp):", pred_orig[0])

Targets: ['CrystalSolub_1', 'Molecular Weight', 'Log P', 'BoilingPoint', 'is_aromatic', 'sigma_780nm', 'max_sigma', 'Tox-score', 'SAscore', 'ISC(S1-T1)']
True (orig, 4dp): [2.095095e+05 1.251700e+02 3.000000e-01 1.786200e+02 0.000000e+00
 7.451000e+01 9.405000e+01 5.900000e-01 2.000000e-02 1.900000e-01]
Pred (orig, 4dp): [7.147738e+04 1.474700e+02 9.100000e-01 2.405500e+02 3.200000e-01
 1.221500e+02 1.519700e+02 6.000000e-01 1.200000e-01 1.200000e-01]


In [50]:
# Assuming your model and scaler are already loaded, and test_dataset is available

# Reinitialize dataset from earlier code
true_values = []
predicted_values = []

for i in range(10):
    d = test_dataset[i].to(device)
    batch_vec = torch.zeros(d.num_nodes, dtype=torch.long, device=device)
    
    with torch.no_grad():
        pred_norm = model(d.x, d.edge_index, batch_vec).cpu().numpy().reshape(1, -1)

    true_norm = d.y.cpu().numpy().reshape(1, -1)

    pred_orig = scaler.inverse_transform(pred_norm)
    true_orig = scaler.inverse_transform(true_norm)

    # Round to 2 decimal places
    pred_orig = np.round(pred_orig, 2)
    true_orig = np.round(true_orig, 2)

    true_values.append(true_orig[0])
    predicted_values.append(pred_orig[0])

# Create a DataFrame
df = pd.DataFrame({
    "Property": TARGET_COLS,
    "True (1st molecule)": true_values[0],
    "Predicted (1st molecule)": predicted_values[0],
})

# Add other molecules (rows 2-10) as columns
for i in range(1, 10):
    temp_df = pd.DataFrame({
        "True ({}th molecule)".format(i+1): true_values[i],
        "Predicted ({}th molecule)".format(i+1): predicted_values[i]
    })
    df = pd.concat([df, temp_df], axis=1)

# Save as CSV
df.to_csv("true_predicted_properties_first_10_molecules.csv", index=False)